In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2664.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2539.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/3217.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/3207.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2673.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2823.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2851.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/3448.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2982.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/3152.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png/2690.png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/V

In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/mohammedjaveed
/kaggle/input/datasets/mohammedjaveed/loveda-dataset
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/masks_png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Rural/images_png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Urban
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Urban/masks_png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Val/Val/Urban/images_png
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Test
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Test/Test
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Test/Test/Rural
/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Test/Test/Rural/images_png
/kaggle/input/datasets/mohammedjaveed/lo

In [3]:
import os

BASE = "/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Train/Train"

URBAN_IMG = f"{BASE}/Urban/images_png"
URBAN_MASK = f"{BASE}/Urban/masks_png"
RURAL_IMG = f"{BASE}/Rural/images_png"
RURAL_MASK = f"{BASE}/Rural/masks_png"

print("Urban images:", len(os.listdir(URBAN_IMG)))
print("Urban masks:", len(os.listdir(URBAN_MASK)))
print("Rural images:", len(os.listdir(RURAL_IMG)))
print("Rural masks:", len(os.listdir(RURAL_MASK)))

Urban images: 1156
Urban masks: 1156
Rural images: 1366
Rural masks: 1366


In [4]:
image_paths = []
mask_paths = []

for name in os.listdir(URBAN_IMG):
    image_paths.append(os.path.join(URBAN_IMG, name))
    mask_paths.append(os.path.join(URBAN_MASK, name))
for name in os.listdir(RURAL_IMG):
    image_paths.append(os.path.join(RURAL_IMG, name))
    mask_paths.append(os.path.join(RURAL_MASK, name))

print("Total images:", len(image_paths))
print("Total masks:", len(mask_paths))

Total images: 2522
Total masks: 2522


In [5]:
import cv2
import numpy as np

mask = cv2.imread(mask_paths[0])
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

print("Mask shape:", mask.shape)
pixels = mask.reshape(-1, 3)
unique_colors = np.unique(pixels, axis=0)

print("Unique colors (first 20):")
print(unique_colors[:20])
print("Total unique colors:", len(unique_colors))

Mask shape: (1024, 1024, 3)
Unique colors (first 20):
[[1 1 1]
 [2 2 2]
 [3 3 3]
 [6 6 6]]
Total unique colors: 4


In [7]:
def process_mask(mask):
    mask = mask[:, :, 0]

    return mask

In [8]:
import torch
from torch.utils.data import Dataset
import cv2

class LoveDADataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = process_mask(mask)
        img = cv2.resize(img, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return img, mask

In [9]:
dataset = LoveDADataset(image_paths, mask_paths)
img, mask = dataset[0]

import torch
print("Mask unique values:", torch.unique(mask))

Mask unique values: tensor([1, 2, 3, 6])


In [10]:
import numpy as np

def remap_mask(mask):
    mapping = {
        1: 0,
        2: 1,
        3: 2,
        6: 3
    }
    new_mask = np.zeros_like(mask)
    for k, v in mapping.items():
        new_mask[mask == k] = v
    return new_mask

In [12]:
class LoveDADataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = process_mask(mask)
        mask = remap_mask(mask)
        img = cv2.resize(img, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()
        return img, mask

In [13]:
dataset = LoveDADataset(image_paths, mask_paths)

img, mask = dataset[0]

import torch
print("Mask unique values:", torch.unique(mask))

Mask unique values: tensor([0, 1, 2, 3])


In [14]:
from torch.utils.data import random_split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 2017
Val: 505


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Loaders ready")

Loaders ready


In [17]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00


In [18]:
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=4   
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
dice_loss = smp.losses.DiceLoss(mode="multiclass")
ce_loss = nn.CrossEntropyLoss()
def combined_loss(pred, target):
    return dice_loss(pred, target) + ce_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Model + Loss ready on:", device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Model + Loss ready on: cuda


In [19]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import os

scaler = GradScaler()

BEST_MODEL_PATH = "/kaggle/working/best_model.pth"

best_val_loss = float("inf")
EPOCHS = 40
patience = 5
counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        imgs = imgs.to(device)
        masks = masks.to(device)

        with autocast():
            preds = model(imgs)
            loss = combined_loss(preds, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Val Epoch {epoch}"):
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = combined_loss(preds, masks)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Best model saved")
        counter = 0
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping")
            break

/tmp/ipykernel_55/3477099562.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Train Epoch 0:   0%|          | 0/253 [00:00<?, ?it/s]/tmp/ipykernel_55/3477099562.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val Epoch 0: 100%|██████████| 64/64 [00:20<00:00,  3.15it/s]



Epoch 0
Train Loss: 1.4293
Val Loss:   1.0389
Best model saved


Val Epoch 1: 100%|██████████| 64/64 [00:13<00:00,  4.70it/s]



Epoch 1
Train Loss: 0.9858
Val Loss:   0.8509
Best model saved


Val Epoch 2: 100%|██████████| 64/64 [00:13<00:00,  4.71it/s]


...


Epoch 17
Train Loss: 0.3959
Val Loss:   0.7086
No improvement (3/5)


Val Epoch 18: 100%|██████████| 64/64 [00:13<00:00,  4.77it/s]



Epoch 18
Train Loss: 0.3767
Val Loss:   0.7013
No improvement (4/5)


Val Epoch 19: 100%|██████████| 64/64 [00:13<00:00,  4.71it/s]


Epoch 19
Train Loss: 0.3484
Val Loss:   0.7210
No improvement (5/5)
Early stopping


In [20]:
import albumentations as A

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(p=0.3),
])

In [22]:
class LoveDADataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = process_mask(mask)
        mask = remap_mask(mask)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]
        img = cv2.resize(img, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return img, mask

In [23]:
dataset = LoveDADataset(image_paths, mask_paths)

from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_indices, val_indices = random_split(range(len(dataset)), [train_size, val_size])

train_dataset = LoveDADataset(
    [image_paths[i] for i in train_indices],
    [mask_paths[i] for i in train_indices],
    transform=transform
)

val_dataset = LoveDADataset(
    [image_paths[i] for i in val_indices],
    [mask_paths[i] for i in val_indices],
    transform=None
)

print(len(train_dataset), len(val_dataset))

2017 505


In [24]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Loaders ready")

Loaders ready


In [25]:
import numpy as np
from tqdm import tqdm

num_classes = 4
class_counts = np.zeros(num_classes)

for _, mask in tqdm(train_dataset):
    mask_np = mask.numpy()
    for i in range(num_classes):
        class_counts[i] += np.sum(mask_np == i)

class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum()

import torch
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)

100%|██████████| 2017/2017 [01:56<00:00, 17.27it/s]


Class weights: tensor([0.0401, 0.2541, 0.5296, 0.1762], device='cuda:0')


In [26]:
import torch.nn as nn
import segmentation_models_pytorch as smp

dice_loss = smp.losses.DiceLoss(mode="multiclass")
ce_loss = nn.CrossEntropyLoss(weight=class_weights)

def combined_loss(pred, target):
    return dice_loss(pred, target) + ce_loss(pred, target)

In [30]:

class LoveDADataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

        mask = process_mask(mask)
        mask = remap_mask(mask)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        img = cv2.resize(img, (384, 384))
        mask = cv2.resize(mask, (384, 384), interpolation=cv2.INTER_NEAREST)

        img = img / 255.0

        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return img, mask

In [32]:
train_dataset = LoveDADataset(
    [image_paths[i] for i in train_indices],
    [mask_paths[i] for i in train_indices],
    transform=transform
)

val_dataset = LoveDADataset(
    [image_paths[i] for i in val_indices],
    [mask_paths[i] for i in val_indices],
    transform=None
)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(len(train_dataset), len(val_dataset))

2017 505


In [33]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import os

scaler = GradScaler()

BEST_MODEL_PATH = "/kaggle/working/best_model.pth"

best_val_loss = float("inf")
EPOCHS = 40

patience = 7
counter = 0

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        imgs = imgs.to(device)
        masks = masks.to(device)

        with autocast():
            preds = model(imgs)
            loss = combined_loss(preds, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Val Epoch {epoch}"):
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = combined_loss(preds, masks)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        counter = 0
        print("saved")
    else:
        counter += 1
        print(counter)

        if counter >= patience:
            print("stop")
            break

/tmp/ipykernel_55/2490788269.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Train Epoch 0:   0%|          | 0/505 [00:00<?, ?it/s]/tmp/ipykernel_55/2490788269.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val Epoch 0: 100%|██████████| 127/127 [00:15<00:00,  8.27it/s]



Epoch 0
Train Loss: 1.2164
Val Loss:   0.9484
saved


Val Epoch 1: 100%|██████████| 127/127 [00:14<00:00,  8.48it/s]



Epoch 1
Train Loss: 1.0191
Val Loss:   0.8003
saved


Val Epoch 2: 100%|██████████| 127/127 [00:14<00:00,  8.50it/s]


...


Epoch 30
Train Loss: 0.6143
Val Loss:   0.7156
5


Val Epoch 31: 100%|██████████| 127/127 [00:15<00:00,  8.31it/s]



Epoch 31
Train Loss: 0.6110
Val Loss:   0.7231
6


Val Epoch 32: 100%|██████████| 127/127 [00:15<00:00,  8.45it/s]


Epoch 32
Train Loss: 0.6019
Val Loss:   0.7204
7
stop
